# Fast advection seeding before directional corridor 4x4 lag-432

This exercise compares three cheap ways to choose a daily advection direction before one full-data Vecchia fit. The final model and optimizer budget are identical across methods; only seed acquisition and the resulting fixed conditioning corridor differ.

1. **pilot400** — first 400 max-min spatial locations per hour, one direction-neutral pointwise Vecchia pilot, two outer L-BFGS calls.
2. **quadrant300** — first 300 max-min locations per hour, four short pilots starting at `(lat, lon) = (0.01, 0.10), (0.01, -0.10), (-0.01, -0.10), (-0.01, 0.10)`; choose the smallest re-evaluated pilot NLL.
3. **empirical** — minimum of the pair-count-filtered, physically smoothed empirical cross-semivariogram at `tau=1`, with ambiguity diagnostics.

The covariance uses `h - v * tau`. Therefore a current target conditions on past clusters near `-v * lag`. The new directional 432 model accepts both signed latitude and longitude advection; the old longitude-only 432 API remains available.

Primary comparison metrics are full-fit Vecchia NLL and end-to-end time. Seed-to-final angle measures stability. Empirical-ridge distance is deliberately not used as a primary score because it would favor the empirical method by construction. Real data have no known wind truth, so a simulation study is still needed for direction accuracy.

In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

LOCAL_REPO = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
AMAREL_REPO = Path('/home/jl2815/tco')
REPO = AMAREL_REPO if AMAREL_REPO.exists() else LOCAL_REPO

DRIVER = REPO / 'Exercises/st_model/day/amarel_simulation/space_time/real_data/advection_direction/compare_advection_seed_methods_corridor432_081626.py'
MODEL_SOURCE = REPO / 'src/GEMS_TCO/vecchia_realdata_corridor_width_4x4_lag432.py'
if not MODEL_SOURCE.exists():
    MODEL_SOURCE = REPO / 'GEMS_TCO/vecchia_realdata_corridor_width_4x4_lag432.py'
RIDGE_SOURCE = REPO / 'GEMS_TCO_EDA/semivariograms/advection_ridge_three_model_compare_081526.ipynb'
RIDGE_PDF = REPO / 'plots/directional_semivariograms/advection_ridge_model_compare_2022_2025_052526/year_2024/advection_ridge_three_models_2024_07_days01_28.pdf'

PYTHON_EXE = Path('/opt/anaconda3/envs/faiss_env/bin/python')
if not PYTHON_EXE.exists():
    PYTHON_EXE = Path(sys.executable)

for path in [DRIVER, MODEL_SOURCE, RIDGE_SOURCE]:
    assert path.exists(), path

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120
print('DRIVER:', DRIVER)
print('MODEL_SOURCE:', MODEL_SOURCE)
print('RIDGE_SOURCE:', RIDGE_SOURCE)
print('RIDGE_PDF:', RIDGE_PDF, 'exists=', RIDGE_PDF.exists())
print('PYTHON_EXE:', PYTHON_EXE)

## Configure

The default is a one-day comparison with execution disabled. Set `RUN_EXPERIMENT=True` after reviewing the printed command. Use `DAYS='0,3'` for day indices 0, 1, 2 and `DAYS='all'` for all 31 days.

In [ ]:
RUN_EXPERIMENT = False
SKIP_EXISTING = True
SUPPRESS_FIT_PRINTS = False

YEARS = ['2024']
MONTH = 7
DAYS = '0'
METHODS = ['pilot400', 'quadrant300', 'empirical']

SPACE = '1,1'
LAT_RANGE = '-3,2'
LON_RANGE = '121,131'
SMOOTH = 0.5
DEVICE = None  # None -> CUDA when available, otherwise CPU

PILOT400_POINTS = 400
QUADRANT300_POINTS = 300
PILOT_NEIGHBORS = 30
PILOT_LIMITS = (20, 20, 20)
PILOT400_OUTER_STEPS = 2
QUADRANT_OUTER_STEPS = 1
PILOT_LBFGS_EVAL = 20
PILOT_LBFGS_HISTORY = 10

EMPIRICAL_TAU = 1
EMPIRICAL_MAX_OFFSETS = (20, 20)
EMPIRICAL_SMOOTH_BANDWIDTH_DEG = 0.063
EMPIRICAL_MIN_PAIR_COUNT = 1_000
EMPIRICAL_NEAR_MIN_REL_TOL = 0.01
EMPIRICAL_NEAR_MIN_ABS_TOL = 0.05
EMPIRICAL_AMBIGUOUS_MIN_NEAR_CELLS = 5

MAX_SEED_NORM = 0.75
FINAL_LBFGS_STEPS = 5
FINAL_LBFGS_EVAL = 20
FINAL_LBFGS_HISTORY = 10
TARGET_CHUNK_SIZE = 128
GRAD_TOL = 1e-5

OUTPUT_ROOT = REPO / 'outputs/day/estimates/advection_seed_three_method_corridor432_081626'
RESULTS_CSV = OUTPUT_ROOT / 'method_results.csv'
CANDIDATES_CSV = OUTPUT_ROOT / 'seed_candidates.csv'
RUN_CONFIG_JSON = OUTPUT_ROOT / 'run_config.json'
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## Verify the new directional geometry

This is a sign-convention check, not a fit. A model seed `v=(0.01, -0.10)` must place its past corridor in the opposite direction `(-0.01, +0.10)`.

In [ ]:
src_dir = MODEL_SOURCE.parent.parent
check_code = (
    "import json, sys; "
    f"sys.path.insert(0, {str(src_dir)!r}); "
    "from GEMS_TCO.vecchia_realdata_corridor_width_4x4_lag432 import directional_model_spec; "
    "print(json.dumps(directional_model_spec(0.01, -0.10)))"
)
example_spec = json.loads(subprocess.check_output([str(PYTHON_EXE), '-c', check_code], text=True))
assert np.isclose(example_spec['past_offset_lat'], -0.01)
assert np.isclose(example_spec['past_offset_lon'], 0.10)
display(pd.Series(example_spec, name='directional lag-432 example'))

## Build and optionally run the experiment

The driver re-evaluates NLL after each short L-BFGS run. This matters for the four-start method because PyTorch L-BFGS returns the loss from the beginning of an outer call, which is not necessarily the loss at the accepted parameters.

In [ ]:
def run_stream(cmd):
    print(shlex.join([str(x) for x in cmd]))
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='')
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f'Command failed with exit code {code}')

cmd = [
    PYTHON_EXE, DRIVER,
    '--years', *YEARS,
    '--month', str(MONTH),
    '--days', DAYS,
    '--methods', *METHODS,
    '--space', SPACE,
    f'--lat-range={LAT_RANGE}',
    f'--lon-range={LON_RANGE}',
    '--smooth', str(SMOOTH),
    '--output-root', OUTPUT_ROOT,
    '--pilot400-points', str(PILOT400_POINTS),
    '--quadrant300-points', str(QUADRANT300_POINTS),
    '--pilot-neighbors', str(PILOT_NEIGHBORS),
    '--pilot-limit-a', str(PILOT_LIMITS[0]),
    '--pilot-limit-b', str(PILOT_LIMITS[1]),
    '--pilot-limit-c', str(PILOT_LIMITS[2]),
    '--pilot400-lbfgs-steps', str(PILOT400_OUTER_STEPS),
    '--quadrant-lbfgs-steps', str(QUADRANT_OUTER_STEPS),
    '--pilot-lbfgs-eval', str(PILOT_LBFGS_EVAL),
    '--pilot-lbfgs-history', str(PILOT_LBFGS_HISTORY),
    '--empirical-tau', str(EMPIRICAL_TAU),
    '--empirical-max-lat-offset', str(EMPIRICAL_MAX_OFFSETS[0]),
    '--empirical-max-lon-offset', str(EMPIRICAL_MAX_OFFSETS[1]),
    '--empirical-smooth-bandwidth-deg', str(EMPIRICAL_SMOOTH_BANDWIDTH_DEG),
    '--empirical-min-pair-count', str(EMPIRICAL_MIN_PAIR_COUNT),
    '--empirical-near-min-rel-tol', str(EMPIRICAL_NEAR_MIN_REL_TOL),
    '--empirical-near-min-abs-tol', str(EMPIRICAL_NEAR_MIN_ABS_TOL),
    '--empirical-ambiguous-min-near-cells', str(EMPIRICAL_AMBIGUOUS_MIN_NEAR_CELLS),
    '--max-seed-norm', str(MAX_SEED_NORM),
    '--final-lbfgs-steps', str(FINAL_LBFGS_STEPS),
    '--final-lbfgs-eval', str(FINAL_LBFGS_EVAL),
    '--final-lbfgs-history', str(FINAL_LBFGS_HISTORY),
    '--target-chunk-size', str(TARGET_CHUNK_SIZE),
    '--grad-tol', str(GRAD_TOL),
]
if DEVICE is not None:
    cmd += ['--device', DEVICE]
if SKIP_EXISTING:
    cmd.append('--skip-existing')
if SUPPRESS_FIT_PRINTS:
    cmd.append('--suppress-fit-prints')

print(shlex.join([str(x) for x in cmd]))
if RUN_EXPERIMENT:
    run_stream(cmd)
else:
    print('RUN_EXPERIMENT=False: command was not executed.')

## Load cached or newly generated results

In [ ]:
results = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
candidates = pd.read_csv(CANDIDATES_CSV) if CANDIDATES_CSV.exists() else pd.DataFrame()

if results.empty:
    print('No result file yet:', RESULTS_CSV)
else:
    ok = results.loc[results['status'].eq('ok')].copy()
    show = [
        'year', 'day', 'method', 'final_nll', 'seed_total_s', 'final_total_s',
        'end_to_end_s', 'seed_lat', 'seed_lon', 'est_advec_lat', 'est_advec_lon',
        'seed_to_est_angle_deg', 'ridge_is_ambiguous',
    ]
    display(ok[[col for col in show if col in ok]].sort_values(['year', 'day', 'final_nll']))
    if len(results) != len(ok):
        display(results.loc[~results['status'].eq('ok'), ['year', 'day', 'method', 'error']])

## Inspect the four-start pilot

`selected=True` is based on the final, explicitly re-evaluated pilot NLL. The four candidates share one precomputed direction-neutral conditioning structure, so the method does not pay that setup cost four times.

In [ ]:
if candidates.empty:
    print('No candidate file yet:', CANDIDATES_CSV)
else:
    candidate_cols = [
        'year', 'day', 'method', 'candidate', 'selected', 'status', 'nll',
        'init_advec_lat', 'init_advec_lon', 'seed_raw_lat', 'seed_raw_lon',
        'fit_s', 'ridge_near_min_count', 'ridge_is_ambiguous', 'ridge_gamma_rel_gap',
    ]
    display(candidates[[col for col in candidate_cols if col in candidates]].sort_values(['year', 'day', 'method', 'nll']))

## Performance comparison

NLL regret is computed within each day relative to that day's best directional corridor. It is more interpretable across days than raw NLL. Runtime includes seed acquisition plus final fit.

In [ ]:
if results.empty or not results['status'].eq('ok').any():
    print('Run or load successful fits first.')
else:
    ok = results.loc[results['status'].eq('ok')].copy()
    ok['nll_regret'] = ok['final_nll'] - ok.groupby(['year', 'day_idx'])['final_nll'].transform('min')
    summary = ok.groupby('method').agg(
        n_days=('final_nll', 'size'),
        median_final_nll=('final_nll', 'median'),
        mean_nll_regret=('nll_regret', 'mean'),
        median_nll_regret=('nll_regret', 'median'),
        median_seed_s=('seed_total_s', 'median'),
        median_final_s=('final_total_s', 'median'),
        median_end_to_end_s=('end_to_end_s', 'median'),
        median_seed_to_est_angle=('seed_to_est_angle_deg', 'median'),
    ).sort_values(['mean_nll_regret', 'median_end_to_end_s'])
    display(summary.round(6))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    sns.boxplot(data=ok, x='method', y='nll_regret', hue='method', legend=False, ax=axes[0])
    sns.stripplot(data=ok, x='method', y='nll_regret', color='black', size=4, alpha=0.55, ax=axes[0])
    axes[0].set_title('Full-fit NLL regret by day')
    axes[0].set_ylabel('NLL - daily best NLL')

    sns.boxplot(data=ok, x='method', y='end_to_end_s', hue='method', legend=False, ax=axes[1])
    sns.stripplot(data=ok, x='method', y='end_to_end_s', color='black', size=4, alpha=0.55, ax=axes[1])
    axes[1].set_title('End-to-end runtime')
    axes[1].set_ylabel('seconds')

    sns.boxplot(data=ok, x='method', y='seed_to_est_angle_deg', hue='method', legend=False, ax=axes[2])
    sns.stripplot(data=ok, x='method', y='seed_to_est_angle_deg', color='black', size=4, alpha=0.55, ax=axes[2])
    axes[2].set_title('Seed-to-final direction movement')
    axes[2].set_ylabel('absolute angle difference (degrees)')
    for ax in axes:
        ax.tick_params(axis='x', rotation=20)
    plt.show()

## Direction plot

Each thin arrow is the seed used to build the corridor; the thick arrow is the final fitted advection. Large rotations indicate that the fixed conditioning corridor and the likelihood optimum disagree.

In [ ]:
if results.empty or not results['status'].eq('ok').any():
    print('Run or load successful fits first.')
else:
    ok = results.loc[results['status'].eq('ok')].copy()
    methods_present = list(ok['method'].drop_duplicates())
    colors = dict(zip(methods_present, sns.color_palette('colorblind', len(methods_present))))
    fig, ax = plt.subplots(figsize=(8, 8), constrained_layout=True)
    for method, group in ok.groupby('method'):
        color = colors[method]
        for _, row in group.iterrows():
            ax.arrow(0, 0, row['seed_lon'], row['seed_lat'], color=color, alpha=0.25, width=0.001, length_includes_head=True)
            ax.arrow(0, 0, row['est_advec_lon'], row['est_advec_lat'], color=color, alpha=0.85, width=0.0025, length_includes_head=True)
        ax.scatter(group['seed_lon'], group['seed_lat'], color=color, marker='x', s=80, label=f'{method} seed')
        ax.scatter(group['est_advec_lon'], group['est_advec_lat'], facecolors='none', edgecolors=[color], marker='o', s=100, label=f'{method} final')
    ax.axhline(0, color='0.5', lw=0.8)
    ax.axvline(0, color='0.5', lw=0.8)
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_xlabel('advection longitude')
    ax.set_ylabel('advection latitude')
    ax.set_title('Seed (x) and final fitted advection (open circle)')
    ax.legend(fontsize=9, ncol=2)
    plt.show()

## How to read the result

A practical winner should have low NLL regret, low end-to-end time, and modest seed-to-final rotation across many days—not just the lowest NLL on one day. Suggested interpretation:

- If the empirical ridge is usually unambiguous and its NLL regret is negligible, it is the natural default because it avoids pilot optimization.
- If empirical surfaces are often ambiguous, `quadrant300` is the safer optimization-based fallback because it explicitly probes all four sign combinations.
- `pilot400` is useful only if the extra 100 points make one-start estimates materially more stable than `quadrant300`; otherwise it offers less protection against a wrong basin.
- Do not rank methods by distance to the empirical ridge: that metric is circular for the empirical method.
- Before production selection, repeat the same comparison on simulated data with known advection and report angular error and norm error in addition to NLL and time.

## Full July command template

After the one-day check succeeds, change `YEARS` and `DAYS` above or use the command below. Existing successful day-method rows can be resumed with `--skip-existing`.

In [ ]:
full_cmd = list(cmd)
year_pos = full_cmd.index('--years')
month_pos = full_cmd.index('--month')
full_cmd[year_pos + 1:month_pos] = ['2022', '2023', '2024', '2025']
days_pos = full_cmd.index('--days')
full_cmd[days_pos + 1] = '0,28'
print(shlex.join([str(x) for x in full_cmd]))